# DRIMC Simulation — Local Notebook Runner

Open multiple copies of this notebook in VS Code and set different `DATASET_IDS` in each to run in parallel.

**Each notebook is responsible for a slice of `dataset_grid`** (i.e. specific `feature_id × repeat_id` combos).  
All train sizes and hyperparameter combos for those datasets are run automatically.

Results are saved to a per-notebook file: `results_drimc_local_{NOTEBOOK_ID}.gz`

## Cell 1 — Configure which datasets this notebook runs

In [1]:
# ============================================================
# USER SETTINGS — change these per notebook instance
# ============================================================

# A unique label for this notebook — used in the output filename
# e.g. 'nb0', 'nb1', 'nb2' ... open one notebook per label
NOTEBOOK_ID = "nb0"

# Which dataset_grid indices (i_ds) this notebook will process.
# dataset_grid has 100 entries (10 feature sizes x 10 repeats),
# indexed 0..99. Split them across notebooks however you like.
#
# Example splits across 4 notebooks:
#   nb0: list(range(0,  25))   # feature_id 0-1, all repeats
#   nb1: list(range(25, 50))   # feature_id 2-3, all repeats
#   nb2: list(range(50, 75))   # feature_id 4-6, all repeats
#   nb3: list(range(75, 100))  # feature_id 7-9, all repeats
#
# Or by repeat_id if you prefer:
#   nb0: [i for i in range(100) if (i % 10) < 5]   # repeat_id 0-4
#   nb1: [i for i in range(100) if (i % 10) >= 5]  # repeat_id 5-9

DATASET_IDS = list(range(0, 50))  # <-- change this per notebook

BASE_SEED = 123456  # must match HPC runs for reproducible splits

# ============================================================

## Cell 2 — Paths

In [ ]:
import os
import sys
import gzip
import pickle
import warnings
import logging

import numpy as np
from tqdm import TqdmSynchronisationWarning, tqdm

warnings.simplefilter("ignore", TqdmSynchronisationWarning)

# ====== user paths — adjust to your local machine ======
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"  # <-- change to local root
PATH_DATA = os.path.join(PATH_ROOT, "datasets/simulations/n_features")
PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/simulations/n_features")
PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")
DRIMC_PATH = os.path.join(PATH_ROOT, "scripts/methods/DRIMC")

for p in (PATH_OUTPUT, PATH_ARCHIVE):
    os.makedirs(p, exist_ok=True)

sys.path.append(PATH_ROOT)

print(f"Output will be saved to: {PATH_OUTPUT}")
print(f"Notebook ID: {NOTEBOOK_ID}  |  Datasets to run: {len(DATASET_IDS)}")

Output will be saved to: /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features
Notebook ID: nb0  |  Datasets to run: 50


## Cell 3 — Imports

In [3]:
from sgimc.utils import mc_split, get_submatrix
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import ParameterGrid, ShuffleSplit, train_test_split
from scipy.special import expit

## Cell 4 — R setup via rpy2

In [4]:
import rpy2.robjects as robjects

r = robjects.r
r(f'setwd("{DRIMC_PATH}")')
print(r("getwd()"))

# load R packages
for pkg in ["matrixcalc", "data.table", "Rcpp", "ROCR", "Bolstad2", "MESS"]:
    r(f'require("{pkg}", character.only=TRUE)')

# source R helper files
for r_file in [
    "doCrossValidationByPairwise.R",
    "doCrossValidationByRow.R",
    "constrNeig.R",
    "inferZeros.R",
    "calcLogLik.R",
    "calcDeriv.R",
    "updateUV.R",
    "calcPredScore.R",
    "calcAUPR.R",
]:
    r(f'source("{r_file}")')

# source C++ extensions
r("library(Rcpp)")
for cpp_file in [
    "fastKF2.cpp",
    "fastKF4.cpp",
    "fastKgipMat.cpp",
    "log1pexp.cpp",
    "sigmoid.cpp",
]:
    r(f'sourceCpp("{cpp_file}")')

updateUV = r["updateUV"]
constrNeig = r["constrNeig"]

print("R setup complete.")

R callback write-console: Loading required package: matrixcalc
  
R callback write-console: Loading required package: data.table
  
R callback write-console: data.table 1.17.8 using 6 threads (see ?getDTthreads).    
R callback write-console: Latest news: r-datatable.com
  
R callback write-console: Loading required package: Rcpp
  
R callback write-console: Loading required package: ROCR
  
R callback write-console: Loading required package: Bolstad2
  
R callback write-console: Loading required package: MESS
  


[1] "/Users/sijianfan/projects/BiSSGL/scripts/methods/DRIMC"

R setup complete.


## Cell 5 — Helper functions

In [5]:
def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]
    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN
    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])
    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])
    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)
    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]


def to_r_matrix(arr):
    return r.matrix(
        robjects.FloatVector(arr.flatten()),
        byrow=True,
        nrow=arr.shape[0],
        ncol=arr.shape[1],
    )


def build_sim_matrices(U, V, K=3):
    """
    Compute Jaccard similarity matrices from U and V,
    then apply neighbourhood constraint via constrNeig().
    Cached per dataset — only recomputed when the dataset changes.
    """
    simD = 1 - pairwise_distances(U, metric="jaccard")
    simT = 1 - pairwise_distances(V, metric="jaccard")
    simD_Robj = to_r_matrix(simD)
    simT_Robj = to_r_matrix(simT)
    lap = constrNeig(simD_Robj, simT_Robj, K=K)
    return simD, simT, lap[2], lap[3]  # (np, np, R, R)


print("Helpers defined.")

Helpers defined.


## Cell 6 — Build grids

In [6]:
n_features_grid = np.arange(50, 501, 50)
n_repeats = 10
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for feature_id, n_features in enumerate(n_features_grid):
    for repeat_id in range(n_repeats):
        dataset_grid.append(
            {
                "feature_id": int(feature_id),
                "n_features": int(n_features),
                "repeat_id": int(repeat_id),
                "filename": os.path.join(
                    PATH_DATA, filename_template.format(n_features, repeat_id)
                ),
            }
        )

grid_dataset = ParameterGrid(
    {
        "train_size": np.arange(0.05, 0.51, 0.05),
        "n_splits": [3],
        "val_size": [0.20],
    }
)

grid_model = ParameterGrid(
    {
        "lamU": [2],
        "lamV": [2],
        "cc": [1, 10],
        "iterpara": [0.125],
        "numLat": [25],
    }
)

# only the datasets assigned to this notebook
my_datasets = [dataset_grid[i] for i in DATASET_IDS]

total = len(my_datasets) * len(list(grid_dataset)) * len(list(grid_model))
print(f"Datasets assigned : {len(my_datasets)}")
print(f"Train size levels : {len(list(grid_dataset))}")
print(f"Hyperparam combos : {len(list(grid_model))}")
print(f"Total combos      : {total}")

Datasets assigned : 50
Train size levels : 10
Hyperparam combos : 2
Total combos      : 1000


## Cell 7 — Run

Progress bars are shown per dataset.  
Results are **incrementally saved** after each dataset finishes — if the kernel dies mid-run you keep completed datasets.

In [7]:
all_results = []
_sim_cache = {}  # cache similarity matrices per (feature_id, repeat_id)

outfile = os.path.join(PATH_OUTPUT, f"results_drimc_local_{NOTEBOOK_ID}.gz")

for i_ds, ds in enumerate(tqdm(my_datasets, desc="Datasets")):

    # ---- load dataset ----
    with gzip.open(ds["filename"], "rb") as fin:
        data = pickle.load(fin)

    U = data["X"]
    V = data["Y"]
    Y = data["R_noisy"].astype(float)
    Y_true = data["R"]

    # ---- similarity matrices (cached) ----
    cache_key = (ds["feature_id"], ds["repeat_id"])
    if cache_key not in _sim_cache:
        print(
            f"  Building sim matrices for feature_id={ds['feature_id']}, repeat_id={ds['repeat_id']}"
        )
        _sim_cache[cache_key] = build_sim_matrices(U, V, K=3)
    simD, simT, simD_Robj, simT_Robj = _sim_cache[cache_key]

    dataset_results = []

    for i_dt, par_dtst in enumerate(grid_dataset):

        # split RNG — same formula as HPC scripts
        split_seed = BASE_SEED + ds["feature_id"] * 1000 + ds["repeat_id"] * 100 + i_dt
        rng_split = np.random.RandomState(split_seed)

        # dev/test split
        dvlp_size, test_size = 0.9, 0.1
        ind_dvlp, ind_test = next(
            mc_split(
                Y,
                n_splits=1,
                random_state=rng_split,
                train_size=dvlp_size,
                test_size=test_size,
            )
        )
        Y_test = get_submatrix(Y_true, ind_test)

        # subsample training indices
        ind_train_all, _ = train_test_split(
            ind_dvlp,
            shuffle=False,
            random_state=rng_split,
            test_size=(1 - (par_dtst["train_size"] / dvlp_size)),
        )

        for i_m, par_mdl in enumerate(grid_model):

            # global combo index — must match HPC indexing for seed consistency
            # i_ds here is the LOCAL index; recover global i_ds from DATASET_IDS
            global_i_ds = DATASET_IDS[i_ds]
            n_dt = len(list(grid_dataset))
            n_m = len(list(grid_model))
            combo_idx = global_i_ds * n_dt * n_m + i_dt * n_m + i_m
            model_seed = BASE_SEED + combo_idx

            lamU = par_mdl["lamU"]
            lamV = par_mdl["lamV"]
            cc = par_mdl["cc"]
            iterpara = par_mdl["iterpara"]
            numLat = par_mdl["numLat"]

            try:
                # ---- full train fit → test score ----
                Y_train_full = get_submatrix(Y, ind_train_all)
                Y_train_full[Y_train_full == -1] = 0.0
                Y_train_full_Robj = to_r_matrix(Y_train_full.toarray())

                out = updateUV(
                    cc=cc,
                    inMat=Y_train_full_Robj,
                    Sd=simD_Robj,
                    St=simT_Robj,
                    lamU=lamU,
                    lamV=lamV,
                    numLat=numLat,
                    initMethod="useSeed",
                    thisSeed=model_seed % (2**31 - 1),
                    iterpara=iterpara,
                    maxIter=1000,
                )
                est_A, est_B = np.array(out[0]), np.array(out[1])

                SA = simD @ est_A
                STB = simT @ est_B
                prob_full = expit(SA @ STB.T)
                prob_test = get_submatrix(prob_full, ind_test)
                scores_test = get_metrics((Y_test.data + 1) / 2, prob_test.data)

                # ---- repeated holdout CV ----
                splt = ShuffleSplit(
                    n_splits=par_dtst["n_splits"],
                    test_size=par_dtst["val_size"],
                    random_state=rng_split,
                )
                for cv, (ind_train, ind_valid) in enumerate(splt.split(ind_train_all)):
                    ind_train_cv = ind_train_all[ind_train]
                    ind_valid_cv = ind_train_all[ind_valid]

                    Y_train = get_submatrix(Y, ind_train_cv)
                    Y_valid = get_submatrix(Y, ind_valid_cv)
                    Y_train[Y_train == -1] = 0.0
                    Y_train_Robj = to_r_matrix(Y_train.toarray())

                    out_cv = updateUV(
                        cc=cc,
                        inMat=Y_train_Robj,
                        Sd=simD_Robj,
                        St=simT_Robj,
                        lamU=lamU,
                        lamV=lamV,
                        numLat=numLat,
                        initMethod="useSeed",
                        thisSeed=model_seed % (2**31 - 1),
                        iterpara=iterpara,
                        maxIter=1000,
                    )
                    est_A_cv, est_B_cv = np.array(out_cv[0]), np.array(out_cv[1])

                    SA_cv = simD @ est_A_cv
                    STB_cv = simT @ est_B_cv
                    prob_full_cv = expit(SA_cv @ STB_cv.T)
                    prob_valid = get_submatrix(prob_full_cv, ind_valid_cv)
                    scores_valid = get_metrics((Y_valid.data + 1) / 2, prob_valid.data)

                    dataset_results.append(
                        {
                            # dataset identity
                            "feature_id": ds["feature_id"],
                            "n_features": ds["n_features"],
                            "repeat_id": ds["repeat_id"],
                            # experimental condition
                            "train_size": par_dtst["train_size"],
                            "n_splits": par_dtst["n_splits"],
                            "val_size": par_dtst["val_size"],
                            # hyperparameters
                            "lamU": lamU,
                            "lamV": lamV,
                            "cc": cc,
                            "iterpara": iterpara,
                            "numLat": numLat,
                            # CV fold
                            "cv": int(cv),
                            # validation scores
                            "val_score": scores_valid,
                            # test scores
                            "test_score": scores_test,
                        }
                    )

            except Exception as e:
                print(
                    f"  ERROR at feature_id={ds['feature_id']}, repeat_id={ds['repeat_id']}, "
                    f"train_size={par_dtst['train_size']:.2f}, "
                    f"lamU={lamU}, lamV={lamV}, cc={cc}: {e}"
                )

    # ---- incremental save after each dataset ----
    all_results.extend(dataset_results)
    with gzip.open(outfile, "wb+", 4) as fout:
        pickle.dump(all_results, fout)
    print(f"  Saved {len(all_results)} rows so far → {outfile}")

print(f"\nDone. Total rows: {len(all_results)}")

Datasets:   0%|          | 0/50 [00:00<?, ?it/s]/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


  Building sim matrices for feature_id=0, repeat_id=0


Datasets:   2%|▏         | 1/50 [02:19<1:53:56, 139.51s/it]

  Saved 60 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_drimc_local_nb0.gz
  Building sim matrices for feature_id=0, repeat_id=1


/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
Datasets:   4%|▍         | 2/50 [04:38<1:51:21, 139.19s/it]

...

  Saved 2880 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_drimc_local_nb0.gz
  Building sim matrices for feature_id=4, repeat_id=8


/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
Datasets:  98%|█████████▊| 49/50 [2:48:44<03:19, 199.19s/it]

  Saved 2940 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_drimc_local_nb0.gz
  Building sim matrices for feature_id=4, repeat_id=9


/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
/Users/sijianfan/.pyenv/versions/BiSSGL_venv_311/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2182: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)
Datasets: 100%|██████████| 50/50 [2:52:05<00:00, 206.52s/it]

  Saved 3000 rows so far → /Users/sijianfan/projects/BiSSGL/outputs/simulations/n_features/results_drimc_local_nb0.gz

Done. Total rows: 3000
